In [1]:
# SNOWFLAKE FUNCTIONS
# Contains all functions related to using snowflake 
import sqlalchemy as db
import pandas as pd


# Connect to my snowflake database.
def connect_snowflake(password):

    # Connection details
    account = "MMBCLIL-IM75488"
    user = "HUGHSULL"
    warehouse = "MY_DEV"
    database = 'RAW'
    schema = "JUPYTER"
    role = "ACCOUNTADMIN"

    
    # Replace the placeholders with your Snowflake credentials and settings
    engine = db.create_engine(
           'snowflake://{user}:{password}@{account}/{database}/{schema}?warehouse={warehouse}&role={role}'.format(
            user= user,
            password= password,
            account= account, 
            database=database,
            schema=schema,
            warehouse=warehouse,
            role=role
        )
    )
    
    # Test connection
    connection = engine.connect()
    print("Connected to Snowflake!")
    connection.close()
    
    return engine



# look at the required skills for a job and get skills ids.
def get_skill_ids(req_skills, connection):
    # Convert the req_skills array into a string format suitable for SQL IN clause
    skills_list_str = "', '".join(req_skills)
    
    # SQL query to get matching skill_ids
    query = f"""
    SELECT skill_id
    FROM skills_list
    WHERE skill_name IN ('{skills_list_str}')
    """
    
    # Fetch the result into a Pandas DataFrame
    df_skill_ids = pd.read_sql(query, connection)
    return df_skill_ids

# get last entry of jobs table ID
def last_job_id(connection):
    # SQL query to get the last job_id
    query = """
    SELECT job_id
    FROM JOBS
    ORDER BY job_id DESC
    LIMIT 1;
    """
    
    # Fetch the result into a Pandas DataFrame
    last_job_id = pd.read_sql(query, connection)

    # return number of rows
    if last_job_id.empty:
        return 0
    else:
        return last_job_id.job_id[0]

# remove duplicated entries in jobs table and then remove associated rows in job_skills table.
def remove_duplicate_entries(connection, password):

    # Pull tables from Snowflake
    jobs_query = "SELECT * FROM jobs"
    job_skills_query = "SELECT * FROM job_skills"
    jobs_df = pd.read_sql(jobs_query, connection)
    job_skills_df = pd.read_sql(job_skills_query, connection)

    # Identify and remove duplicates in jobs_df
    dup_columns = ['title', 'company', 'location', 'employment_type', 'salary', 'pay_period', 'post_date']
    
    # Identify the duplicate rows
    duplicates = jobs_df[jobs_df.duplicated(subset=dup_columns, keep='first')]
    
    # Get the job_ids of the duplicates
    removed_job_ids = duplicates['job_id'].tolist() # list of removed job_ids
    removed_job_ids_str = list(map(str, removed_job_ids))
    
    # Remove duplicates from jobs_df 
    jobs_df_clean = jobs_df.drop_duplicates(subset=dup_columns, keep='first')
    
    # Step 2: Remove corresponding rows from job_skills_df based on job_ids
    job_skills_df_clean = job_skills_df[~job_skills_df['job_id'].isin(removed_job_ids_str)]

    # Replace the cleaned data back in Snowflake 
    jobs_df_clean.to_sql('jobs', con=connection, if_exists='replace', index=False)
    job_skills_df_clean.to_sql('job_skills', con=connection, if_exists='replace', index=False)
    print(f"{len(removed_job_ids)} duplicate job advertisements removed.")

# get Snowflake tables
def get_snowflake_tables(connection):
    jobs_df = pd.read_sql("SELECT * FROM jobs", connection)  
    job_skills_df = pd.read_sql("SELECT * FROM job_skills", connection)
    skills_df = pd.read_sql("SELECT * FROM skills_list", connection)
    

    return jobs_df, job_skills_df, skills_df